# 01 · Data Cleaning

This notebook loads `borrower_cohort.csv` (exported from the SQL `borrower_cohort` view), splits it into train / validation / test, and cleans it. Every cleaning value (medians, caps) is learned from the **training set only** and then applied to validation and test, to avoid data leakage.

## 1. Imports and data load

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

path = '../data/'

df = pd.read_csv(path + 'borrower_cohort.csv', na_values=['\\N', 'NULL', ''])

print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nNull counts:\n", df.isnull().sum())

Shape: (307511, 20)

Data types:
 SK_ID_CURR              int64
TARGET                  int64
age_years               int64
employment_years      float64
annual_income         float64
loan_amount           float64
annual_repayment      float64
debt_to_income        float64
repayment_burden      float64
gender                 object
employment_type        object
education              object
family_status          object
owns_car               object
owns_property          object
num_children            int64
region_risk_rating      int64
EXT_SOURCE_1          float64
EXT_SOURCE_2          float64
EXT_SOURCE_3          float64
dtype: object

Null counts:
 SK_ID_CURR                0
TARGET                    0
age_years                 0
employment_years      55374
annual_income             0
loan_amount               0
annual_repayment          0
debt_to_income            0
repayment_burden          0
gender                    0
employment_type           0
education                 0
f

## 2. Train / validation / test split (70 / 15 / 15)

The test set is split off first, then the rest is split into train and validation. The split is stratified on `TARGET`, so all three sets keep the same ~8% default rate. Everything after this point is fitted on train only.

In [5]:
train_val, test = train_test_split(
    df, test_size=0.15, stratify=df['TARGET'], random_state=42
)

train, val = train_test_split(
    train_val, test_size=0.15 / 0.85, stratify=train_val['TARGET'], random_state=42
)

print("Train:", train.shape, " Val:", val.shape, " Test:", test.shape)
print("Default rate - train:", round(train['TARGET'].mean() * 100, 2))
print("Default rate - val  :", round(val['TARGET'].mean() * 100, 2))
print("Default rate - test :", round(test['TARGET'].mean() * 100, 2))

Train: (215257, 20)  Val: (46127, 20)  Test: (46127, 20)
Default rate - train: 8.07
Default rate - val  : 8.07
Default rate - test : 8.07


## 3. Raw EXT_SOURCE copy

When the data was loaded into MySQL, blank EXT_SOURCE values were stored as 0. These are converted back to missing (NaN). A `_raw` copy with the real gaps is kept, because XGBoost can learn from missing values directly.

In [7]:
ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

for data in (train, val, test):
    for col in ext_cols:
        data[col] = data[col].replace(0, np.nan)
        data[col + '_raw'] = data[col]

## 4. Employment years: flag, then impute

Missing `employment_years` comes from the 365243 placeholder in the raw data, which marks people with no current job (almost all pensioners). A `not_employed_flag` is created **before** filling the gaps, so this information isn't lost. The gaps are then filled with the training median.

In [9]:
for data in (train, val, test):
    data['not_employed_flag'] = data['employment_years'].isna().astype(int)

print("Not-employed flag - share of train:", round(train['not_employed_flag'].mean() * 100, 2), "%")
print(pd.crosstab(train['employment_type'], train['not_employed_flag']))
print("\nDefault rate by flag:\n", train.groupby('not_employed_flag')['TARGET'].mean().round(4))

employment_median = train['employment_years'].median()

for data in (train, val, test):
    data['employment_years'] = data['employment_years'].fillna(employment_median)

Not-employed flag - share of train: 17.94 %
not_employed_flag          0      1
employment_type                    
Businessman                9      0
Commercial associate   50352      0
Maternity leave            2      0
Pensioner                  7  38600
State servant          15226      0
Student                   11      0
Unemployed                 0     18
Working               111032      0

Default rate by flag:
 not_employed_flag
0    0.0866
1    0.0539
Name: TARGET, dtype: float64


## 5. EXT_SOURCE imputation

Logistic regression and random forest can't accept missing values, so the EXT_SOURCE columns are filled with the training median for each employment type, falling back to the overall training median.

In [11]:
overall_medians = train[ext_cols].median()

for col in ext_cols:
    group_medians = train.groupby('employment_type')[col].median()

    for data in (train, val, test):
        fill_values = data['employment_type'].map(group_medians)
        data[col] = data[col].fillna(fill_values)
        data[col] = data[col].fillna(overall_medians[col])

print("Remaining nulls after imputation:")
print("Train:", train[ext_cols + ['employment_years']].isnull().sum().sum())
print("Val  :", val[ext_cols + ['employment_years']].isnull().sum().sum())
print("Test :", test[ext_cols + ['employment_years']].isnull().sum().sum())

Remaining nulls after imputation:
Train: 0
Val  : 0
Test : 0


## 6. Outlier capping

Income, loan amount and the loan-to-income ratio (named `debt_to_income` in the code) have long right tails. Values are capped at the training 99.5th percentile, which limits extreme values without deleting any borrowers.

In [13]:
print("Train percentiles - annual_income:")
print(train['annual_income'].quantile([0.90, 0.95, 0.99, 0.995, 1.00]))

print("\nTrain percentiles - loan_amount:")
print(train['loan_amount'].quantile([0.90, 0.95, 0.99, 0.995, 1.00]))

print("\nTrain percentiles - debt_to_income:")
print(train['debt_to_income'].quantile([0.90, 0.95, 0.99, 0.995, 1.00]))

Train percentiles - annual_income:
0.900       270000.0
0.950       337500.0
0.990       472500.0
0.995       630000.0
1.000    117000000.0
Name: annual_income, dtype: float64

Train percentiles - loan_amount:
0.900    1134000.0
0.950    1350000.0
0.990    1864152.0
0.995    2013840.0
1.000    4050000.0
Name: loan_amount, dtype: float64

Train percentiles - debt_to_income:
0.900     7.486
0.950     9.143
0.990    13.045
0.995    14.838
1.000    49.227
Name: debt_to_income, dtype: float64


In [14]:
income_cap = train['annual_income'].quantile(0.995)
loan_cap = train['loan_amount'].quantile(0.995)
dti_cap = train['debt_to_income'].quantile(0.995)

print("Caps fit on train - income:", income_cap, " loan:", loan_cap, " dti:", dti_cap)

for data in (train, val, test):
    data['annual_income'] = data['annual_income'].clip(upper=income_cap)
    data['loan_amount'] = data['loan_amount'].clip(upper=loan_cap)
    data['debt_to_income'] = data['debt_to_income'].clip(upper=dti_cap)

Caps fit on train - income: 630000.0  loan: 2013840.0  dti: 14.83800000000003


## 7. Clean categorical columns

In [16]:
for data in (train, val, test):
    for col in data.select_dtypes(include=['object', 'string']).columns:
        data[col] = data[col].str.strip()

print("Gender unique values:", train['gender'].unique())
print("Employment type unique values:", train['employment_type'].unique())
print("Education unique values:", train['education'].unique())
print("Family status unique values:", train['family_status'].unique())
print("Owns car unique values:", train['owns_car'].unique())
print("Owns property unique values:", train['owns_property'].unique())

Gender unique values: ['M' 'F' 'XNA']
Employment type unique values: ['Commercial associate' 'Working' 'Pensioner' 'State servant' 'Student'
 'Unemployed' 'Businessman' 'Maternity leave']
Education unique values: ['Higher education' 'Secondary / secondary special' 'Incomplete higher'
 'Lower secondary' 'Academic degree']
Family status unique values: ['Married' 'Single / not married' 'Civil marriage' 'Widow' 'Separated'
 'Unknown']
Owns car unique values: ['Y' 'N']
Owns property unique values: ['Y' 'N']


## 8. Save cleaned files

In [18]:
train.to_csv(path + 'borrower_clean_train.csv', index=False)
val.to_csv(path + 'borrower_clean_val.csv', index=False)
test.to_csv(path + 'borrower_clean_test.csv', index=False)

print(train.columns.tolist())

['SK_ID_CURR', 'TARGET', 'age_years', 'employment_years', 'annual_income', 'loan_amount', 'annual_repayment', 'debt_to_income', 'repayment_burden', 'gender', 'employment_type', 'education', 'family_status', 'owns_car', 'owns_property', 'num_children', 'region_risk_rating', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'EXT_SOURCE_1_raw', 'EXT_SOURCE_2_raw', 'EXT_SOURCE_3_raw', 'not_employed_flag']
